**Name:**

**Matrikel Nummer:**

# Fake news detection

*(For political and gossip data)*

## Data Cleaning

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re
from urllib.parse import urljoin
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from collections import Counter
from nltk.stem.snowball import SnowballStemmer
import collections
import string
import nltk
import pickle as pkl
import tldextract

### Unifying Columns names

Because we have many data sources, so we unified the name of the most important columns and make accessing to the data easier

In [2]:
#unified name columns
def Unified_name(data,names):
    return data.rename(columns = {names[0]:'text',names[1]:'url',names[2]:'label'})

### Get the important columns

We only need three columns:
- 'text': which is the whole text of the news
- 'url' : The page that we got the news from
- 'label' : That define the news is Fake or Real

From these three columns we will extract some other columns

In [3]:
def get_important_columns(data):
    return data[['text','url','label']]

### Get the Domain Name

This function is created to get the domain name from the url columns

In [4]:
#get the domain address from the full url
def get_domain(data):
    data['source'] = [tldextract.extract(text).domain for text in data['url'].values]

### Clean the degits

This function is created to clean the digits from the texts

In [5]:
#remove the digits from the texts
def clean_digit(text):
    result = re.sub(r'\d+', '', text)
    return result

### Remove Stop words

This function is created to remove stop words from the texts

In [6]:
def remove_stops_punct(text):
    text_no_punct = text.translate(str.maketrans('', '', string.punctuation))
    stop_words = set(stopwords.words('english'))
    word_tokens = word_tokenize(text_no_punct)
    text_no_stops = [w for w in word_tokens if not w in stop_words]
    return text_no_stops

In [7]:
stm = SnowballStemmer("english")
def stemmer(l):
    stemmed =[]
    for w in l:
        if len(w) > 2:
            stemmed.append(stm.stem(w))
    return stemmed

In [8]:
#function count the tokens of the body
def countoftokens(l):
    return len(l)

### Cleaning the text

In [9]:
#function clean the text
def clean_text(data_set):
    #convert the text to the lower case and store it in clean
    data_set['clean'] = data_set['text'].str.lower()
    #clean the digit
    data_set['clean'] = data_set['clean'].apply(clean_digit)
    #replace the new lines with blanks
    data_set['clean'] = data_set['clean'].str.replace('\n','')
    #split the text into scentences and put it in column scentences
    data_set['sentences']=data_set['clean'].str.split('.')

### Remove the data from famous websites

In [10]:
#remove obvious websites by using the column source that we created
def remove_useless_row(data_set):
    data_set=data_set[data_set['source']!='https://twitter.com']
    data_set=data_set[data_set['source']!='http://books.google.com']
    data_set=data_set[data_set['source']!='http://en.wiktionary.org']
    data_set=data_set[data_set['source']!='http://www.youtube.com']
    data_set=data_set[data_set['source']!='http://youtube.com']
    data_set=data_set[data_set['source']!='https://youtu.be']

## Unified DataFrames

Unifying the Dataframes that contains different data sources are done by:
- Unifying the name of the columns
- Unifying the labeling way for the types of news
- Get the most important columns

The datasets are ready to be merged after we do the mentioned steps

In [11]:
df=pd.read_csv('data.csv')
df=Unified_name(df,['Body','URLs','Label'])
df=get_important_columns(df)
# replace the labeling value to -1 as real and 1 as fake
df.label=df.label.replace(1,-1)
df.label=df.label.replace(0,1)
print("The First Dataset:")
print(df.columns)
print(df.shape)
print('________________________________________________')

df2=pd.read_csv('news_sample.csv')
df2=Unified_name(df2,['content','url','type'])
df2=get_important_columns(df2)
df2= df2[df2.label != 'unknown']
df2= df2[df2.label != 'political']
df2.label=df2.label.replace('reliable',-1)
df2.label[df2.label!=-1]=1
print("The Second Dataset:")
print(df2.columns)
print(df2.shape)
print('________________________________________________')


df3=pd.read_pickle('dataset.pkl')
df3=Unified_name(df3,['text','url','label'])
df3=get_important_columns(df3)
df3.label=df3.label.replace('real',-1)
df3.label=df3.label.replace('fake',1)
print("The Third Dataset:")
print(df3.columns)
print(df3.shape)
print('________________________________________________')

df4=pd.read_pickle('dataset_2.pkl')
df4=df4.drop(columns='url')
df4=Unified_name(df4,['text','url_source','label'])
df4=get_important_columns(df4)
df4.label=-1
print("The Fourth Dataset:")
print(df4.columns)
print(df4.shape)

The First Dataset:
Index(['text', 'url', 'label'], dtype='object')
(4009, 3)
________________________________________________
The Second Dataset:
Index(['text', 'url', 'label'], dtype='object')
(221, 3)
________________________________________________
The Third Dataset:
Index(['text', 'url', 'label'], dtype='object')
(4147, 3)
________________________________________________
The Fourth Dataset:
Index(['text', 'url', 'label'], dtype='object')
(1613, 3)


## Merge DataFrames

we merge the dataframe and make it under one dataframe to make the process of cleaning data once.

In [12]:
Data=pd.DataFrame()
Data=Data.append(df,sort=True)
Data=Data.append(df2,sort=True)
Data=Data.append(df3,sort=True)
Data=Data.append(df4,sort=True)

Data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 9990 entries, 0 to 1930
Data columns (total 3 columns):
label    9990 non-null object
text     9969 non-null object
url      9990 non-null object
dtypes: object(3)
memory usage: 312.2+ KB


## Start Cleaning Data

Start Cleaning Data by:
- Dropping the columns that contains Nan Values
- Dropping Duplicates Columns
- Utilizing the functions of cleaning that we created

Finally, we reindex the dataframe.

In [13]:
Data=Data.dropna(how='any')

Data=Data.drop_duplicates(subset ="text",keep = 'first')


#unified the name of the columns name ( because we have many data sources)
get_domain(Data)

#call the function
clean_text(Data)


#creat the tokens column and put it in tokens columns
Data['tokens']= Data['clean'].apply(remove_stops_punct)
#apply Stemmer
Data['tokens'] = Data['tokens'].apply(stemmer)
#count the tokens for each row and put it in token_count column
Data['tokens_count'] = Data['tokens'].apply(countoftokens)
#call the function
remove_useless_row(Data)

#remove the rows where tokens less than 30
Data=Data[Data.tokens_count>30]

Data = Data.reset_index()
Data=Data.drop(['index'],axis=1)

### Save the Data as .pkl file

In [15]:
pkl.dump( Data, open( "cleandata.pkl", "wb" ) )
Data.head()

label  ...  sent
0     -1  ...  ...
1     -1  ...  ...
2     -1  ...  ...
3     -1  ...  ...
4      1  ...  ...

### Read the data from the saved file

In [16]:
Data=pd.read_pickle("cleandata.pkl")

### Plot Frequencies of Labels

We create these to plots to show that balancing of the data

In [25]:
plt.figure()
fig, axes = plt.subplots(1, 2, figsize=(12,6))
fig.suptitle('Frequencies of news types', fontsize=20)

count_Class=pd.value_counts(Data["label"], sort= True)

a=sns.countplot(x="label", data=Data, ax=axes[0],order = Data.label.value_counts().index)
a.set_xticklabels(['Fake News','Real News'])

axes[1].pie(count_Class, startangle=90,labels=['Fake News','Real News'], autopct='%1.0f%%')
axes[0].set_title('Labels Frequencies')
axes[1].set_title('Percentage of Classes')
plt.subplots_adjust(hspace=10)

<Figure size 432x288 with 0 Axes>


### Frequency of Tokens in the clear text and raw text

In this section we show the frequencies of all words in all texts for both kind of text:
- The cleaning texts (The processed)
- The raw texts (The Original)
  To show the amounts of usless words in the texts.

In [26]:
#tc: Tokens Counts
#For Clear Text
#Create Dectionary of words and frequencies
count_token_clear = Counter(np.concatenate(Data.tokens.values))
#convert the dictionary to dataframe
tc_clear = pd.DataFrame.from_dict(count_token_clear, orient='index')
#make the column name count word
tc_clear.rename(columns={0: 'countword'}, inplace=True)
#sort the values according to the frequencies
tc_clear=tc_clear.sort_values(by='countword',ascending=False)
#________________________________________________

#Create Dectionary of words and frequencies
count_token_notclear = Counter(np.concatenate(Data['text'].apply(word_tokenize).values))
#convert the dictionary to dataframe
tc_notclear = pd.DataFrame.from_dict(count_token_notclear, orient='index')
#make the column name count word
tc_notclear.rename(columns={0: 'countword'}, inplace=True)
#sort the values according to the frequencies
tc_notclear=tc_notclear.sort_values(by='countword',ascending=False)

### Plot The Frequencies of the token for Clean Text and Raw Text

Here, we visualize the words frequencies in all texts in the data (for the raw texts and clear ones).

In [13]:
plt.figure()

fig, axes = plt.subplots(1, 2, figsize=(14,10))
fig.suptitle('Raw Text                    VS                    Clear Text', fontsize=14)


p=sns.barplot(tc_clear.countword.values[0:40],tc_clear.index.values[0:40],ax=axes[1])
p.xaxis.tick_top()


p2=sns.barplot(tc_notclear.countword.values[0:40],tc_notclear.index.values[0:40],ax=axes[0])
p2.xaxis.tick_top()

plt.show()

<Figure size 432x288 with 0 Axes>
